# Lab 13 — Structured Streaming (Rate Source)

**Tujuan:** Mengamati micro-batches, checkpoint, output mode, dan query lifecycle.

**Lingkungan:** Spark standalone (`spark-master:7077`), Jupyter, dua worker. Jalankan sel secara berurutan. Setiap notebook dapat dijalankan sendiri.

**Metode:** Prediksi → jalankan → amati `explain()`/Spark UI → ubah satu variabel → simpulkan.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Spark Training - Lab 13")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

## 1. Use a built-in streaming source

Sumber `rate` tidak membutuhkan producer Kafka atau paket tambahan. **Tidak ada kode yang mengubah Compose.**

In [ ]:
stream = spark.readStream.format("rate").option("rowsPerSecond", 5).load()
print("Streaming?", stream.isStreaming)
stream.printSchema()

## 2. Start streaming into an in-memory sink

Nama query dibuat unik; stop query di bawah sebelum menjalankan ulang notebook. Checkpoint tidak diperlukan untuk demo non-fault-tolerant memory sink ini.

In [ ]:
import time
result = stream.withColumn("group_id", F.col("value") % 3)
query = (
    result.writeStream
    .format("memory")
    .queryName("lab13_events")
    .outputMode("append")
    .start()
)
time.sleep(8)
print("Active:", query.isActive)
print("Progress:", query.lastProgress)

## 3. Query a streaming materialization and stop

In [ ]:
spark.sql("SELECT group_id, COUNT(*) AS rows FROM lab13_events GROUP BY group_id").show()
query.stop()
print("Stopped:", not query.isActive)

**Tugas:** cari batch number, numInputRows, dan duration di `lastProgress`. Jelaskan mengapa streaming DataFrame tidak dapat langsung menjalankan `show()` sebelum ditulis melalui streaming sink.

## Refleksi peserta

Tuliskan: **(1)** apa prediksi Anda, **(2)** apa yang teramati di Spark UI atau output, **(3)** mengapa Spark berperilaku demikian. Hindari menyimpulkan dari waktu eksekusi satu kali saja.